In [1]:
import pandas as pd

df = pd.read_json("processed_df.jsonl", lines=True)


In [4]:
df.columns

Index(['text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'entity_type'],
      dtype='object')

In [2]:
df.loc[df['uuid'] == '7aab7d3a-8748-406e-b0bf-0d191bf5e0da']

,text-topic,text-topic-exp,text-entity-name,text-entity-gender,text-entity-sentiment,text-entity-sentiment-exp,text-generic-frame,text-generic-frame-exp,text-issue-frame,text-issue-frame-exp,...,description,language,maintext,source_domain,url,political_leaning,month,keyword_matches,keyword_match_count,entity_type
5848,Campus Protests for Divestment from Israel,"The article discusses ongoing campus protests,...",Pro-Palestinian protests,None,none,The article does not provide sufficient inform...,"[Political, Fairness and equality, Cultural id...",The article discusses political actions (prote...,Political Conflict,"The article discusses ongoing campus protests,...",...,More arrests were made on college campuses Sat...,en,Nationwide campus protests calling for divestm...,www.cbsnews.com,https://www.cbsnews.com/chicago/video/nationwi...,left_lean,2024-04-01_2024-04-30,"[protests, protests]",2,NORP


# We focus on annotating 40 articles from the 15k regex-filtered dataset, matches >= 2. We compute simpledorff score

In [ ]:
rel_sample = df.sample(n=40, random_state = 7)

rel_export = rel_sample[['uuid', 'title', 'maintext']]

rel_export.to_csv("relevance_sample_1.csv", index=False)

In [2]:
rel_sample2 = df.sample(n=40, random_state = 8)

rel_export2 = rel_sample2[['uuid', 'title', 'maintext']]

rel_export2.to_csv("relevance_sample2.csv", index=False)

In [4]:
import pandas as pd

df = pd.read_csv("relevance_scores.csv")
df.head()



,Jeremy,Julia,jeremy_3,julia_3,jeremy_binary,julia_binary
0,irrelevant,irrelevant,0,0,0,0
1,relevant_indirect,relevant_indirect,1,1,1,1
2,irrelevant,irrelevant,0,0,0,0
3,relevant_direct,relevant_direct,2,2,1,1
4,relevant_indirect,relevant_indirect,1,1,1,1


In [ ]:
import pandas as pd
from simpledorff.simpledorff import calculate_krippendorffs_alpha
from simpledorff import data_transforms


def to_ea_table(long_df: pd.DataFrame,
                experiment_col: str,
                annotator_col: str,
                value_col: str) -> pd.DataFrame:
    """
    Convert a long-form df (unit, annotator, value) to simpledorff's
    experiment-annotator table, handling multiple simpledorff versions.
    """
    fn = data_transforms.df_to_experiment_annotator_table

    try:
        return fn(long_df, experiment_col, annotator_col, value_col)
    except TypeError:
        pass

    

def kripp_alpha_simpledorff(df: pd.DataFrame, col_a: str, col_b: str) -> float:
    """
    Compute Krippendorff's alpha between two coder columns using older simpledorff versions
    that require an experiment-annotator table.
    """
    sub = df[[col_a, col_b]].copy()
    sub[col_a] = pd.to_numeric(sub[col_a], errors="coerce")
    sub[col_b] = pd.to_numeric(sub[col_b], errors="coerce")

    # Create unit id
    sub = sub.reset_index(drop=True)
    sub["unit"] = sub.index

    # Long format: unit / annotator / value
    long_df = sub.melt(
        id_vars="unit",
        value_vars=[col_a, col_b],
        var_name="annotator",
        value_name="value"
    ).dropna(subset=["value"])

    ea_table = to_ea_table(long_df, "unit", "annotator", "value")

    # nominal metric is default in your simpledorff
    return float(calculate_krippendorffs_alpha(ea_table))


# ---- Run on your CSV ----
df = pd.read_csv("relevance_scores.csv")

alpha_3class = kripp_alpha_simpledorff(df, "jeremy_3", "julia_3")
alpha_binary = kripp_alpha_simpledorff(df, "jeremy_binary", "julia_binary")

print("Krippendorff's alpha (3-class):", alpha_3class)
print("Krippendorff's alpha (binary):", alpha_binary)


Krippendorff's alpha (3-class): 0.7390278433223219
Krippendorff's alpha (binary): 0.9425454545454546
